In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Main table from your screenshot
TABLE_NAME = "workspace.default.gold_user_product_interactions"

df = spark.table(TABLE_NAME)

display(df.limit(10))
print("Rows:", df.count())
print("Columns:", len(df.columns))
print(df.columns)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# SETTINGS
TEST_FRAC = 0.2
SEED = 42
MIN_USER_REVIEWS = 30
MIN_ITEM_REVIEWS = 120
#MIN_USER_REVIEWS = 5
#MIN_ITEM_REVIEWS = 20
MIN_CO_RATINGS = 5
#MIN_CO_RATINGS = 3
TOP_K_NEIGHBORS = 10
#TOP_K_NEIGHBORS = 30
MIN_SIMILARITY = 0.05

#Is_recommended & ratings threshold have been hard coded into 1 & 5 respectively in the code
HOLDOUT_TABLE = "workspace.default.sephora_test_holdout"

print("Configured per-user split:", int((1-TEST_FRAC)*100), "/", int(TEST_FRAC*100))
print("Min user reviews:", MIN_USER_REVIEWS)
print("Min item reviews:", MIN_ITEM_REVIEWS)
print("Min co-ratings:", MIN_CO_RATINGS)
print("Top-K neighbors:", TOP_K_NEIGHBORS)


In [0]:
print("Holdout table will be written after the train/test split is created:", HOLDOUT_TABLE)


In [0]:
ratings_raw = (
    df.select(
        F.col("user_id").cast("string").alias("user_id"),
        F.col("product_id").cast("string").alias("item_id"),
        F.col("rating").cast("double").alias("rating")
    )
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("item_id").isNotNull())
    .filter(F.col("rating").isNotNull())
)

display(ratings_raw.limit(10))
print("Ratings rows:", ratings_raw.count())

In [0]:
# Remove impossible ratings and duplicate user-item pairs
# If a user reviewed the same product multiple times, keep the mean rating
ratings_clean = (
    ratings_raw
    .filter((F.col("rating") >= 1.0) & (F.col("rating") <= 5.0))
    .groupBy("user_id", "item_id")
    .agg(
        F.avg("rating").alias("rating")
    )
)

display(ratings_clean.limit(10))
print("Clean ratings rows:", ratings_clean.count())

In [0]:
# Minimum activity thresholds are applied BEFORE splitting
active_users = (
    ratings_clean.groupBy("user_id")
    .count()
    .filter(F.col("count") >= MIN_USER_REVIEWS)
    .select("user_id")
)

popular_items = (
    ratings_clean.groupBy("item_id")
    .count()
    .filter(F.col("count") >= MIN_ITEM_REVIEWS)
    .select("item_id")
)

ratings_filtered = (
    ratings_clean.join(active_users, on="user_id", how="inner")
    .join(popular_items, on="item_id", how="inner")
)

# Per-user 80/20 random split so every user keeps history in train
w_split = Window.partitionBy("user_id").orderBy(F.rand(SEED))

ratings_split = (
    ratings_filtered
    .withColumn("row_num", F.row_number().over(w_split))
    .withColumn("user_count", F.count("*").over(Window.partitionBy("user_id")))
    .withColumn(
        "test_cutoff",
        F.when(
            F.col("user_count") >= 2,
            F.greatest(F.lit(1), F.floor(F.col("user_count") * F.lit(TEST_FRAC)).cast("int"))
        ).otherwise(F.lit(0))
    )
    .withColumn("is_test", F.col("row_num") <= F.col("test_cutoff"))
)

ratings_train = (
    ratings_split
    .filter(~F.col("is_test"))
    .drop("row_num", "user_count", "test_cutoff", "is_test")
)

ratings_test = (
    ratings_split
    .filter(F.col("is_test"))
    .drop("row_num", "user_count", "test_cutoff", "is_test")
)

# Continue the notebook using TRAIN only
ratings = ratings_train
spark.sql(f"DROP TABLE IF EXISTS {HOLDOUT_TABLE}")
ratings_test.write.format("delta").mode("overwrite").saveAsTable(HOLDOUT_TABLE)

print("Filtered ratings rows (full eligible set):", ratings_filtered.count())
print("Train rows:", ratings_train.count())
print("Test rows:", ratings_test.count())
print("Train users:", ratings_train.select("user_id").distinct().count())
print("Test users:", ratings_test.select("user_id").distinct().count())
print("Train items:", ratings_train.select("item_id").distinct().count())
print("Test items:", ratings_test.select("item_id").distinct().count())

display(ratings.limit(10))
display(ratings_test.limit(10))


In [0]:
display(ratings_raw.limit(10))

In [0]:
from pyspark.sql import functions as F, Window

# ---- build manual integer index lookups ----

user_index_lookup = (
    ratings_filtered
    .select("user_id")
    .dropDuplicates(["user_id"])
    .withColumn(
        "user_index",
        F.row_number().over(Window.orderBy("user_id")) - 1
    )
    .select("user_id", F.col("user_index").cast("int"))
)

item_index_lookup = (
    ratings_filtered
    .select("item_id")
    .dropDuplicates(["item_id"])
    .withColumn(
        "item_index",
        F.row_number().over(Window.orderBy("item_id")) - 1
    )
    .select(F.col("item_index").cast("int"), "item_id")
)

ratings_train_idx = (
    ratings_train
    .join(user_index_lookup, on="user_id", how="inner")
    .join(item_index_lookup, on="item_id", how="inner")
)

ratings_test_idx = (
    ratings_test
    .join(user_index_lookup, on="user_id", how="inner")
    .join(item_index_lookup, on="item_id", how="inner")
)

display(ratings_train_idx.limit(10))
display(ratings_test_idx.limit(10))

In [0]:
# ==========================================
# CREATE ITEM NAME LOOKUP
# ==========================================

product_features = spark.table("workspace.default.gold_product_features")

item_name_lookup = (
    item_index_lookup
    .join(
        product_features.select(
            F.col("product_id").cast("string").alias("item_id"),
            F.col("product_name")
        ),
        on="item_id",
        how="left"
    )
    .select("item_id", "item_index", "product_name")
)

display(item_name_lookup.limit(10))

In [0]:
from pyspark.ml.recommendation import ALS

ALS_RANK = 20
ALS_MAX_ITER = 10
ALS_REG_PARAM = 0.1
ALS_NONNEGATIVE = True

als = ALS(
    userCol="user_index",
    itemCol="item_index",
    ratingCol="rating",
    rank=ALS_RANK,
    maxIter=ALS_MAX_ITER,
    regParam=ALS_REG_PARAM,
    nonnegative=ALS_NONNEGATIVE,
    implicitPrefs=False,
    coldStartStrategy="drop",
    seed=SEED
)

als_model = als.fit(ratings_train_idx)

print("ALS rank:", ALS_RANK)
print("ALS maxIter:", ALS_MAX_ITER)
print("ALS regParam:", ALS_REG_PARAM)
print("ALS nonnegative:", ALS_NONNEGATIVE)

In [0]:
#HYPER PARAMETER TUNING WITH GRID SEARCH
import itertools
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.sql import functions as F, Window

# Minimal grid for Databricks Free / serverless
configs = [
    {"rank": 10, "regParam": 0.05},
    {"rank": 10, "regParam": 0.10},
    {"rank": 20, "regParam": 0.05},
    {"rank": 20, "regParam": 0.10},
    {"rank": 30, "regParam": 0.05},
    {"rank": 30, "regParam": 0.10},
]

ALS_MAX_ITER_TUNE = 8
ALS_NONNEGATIVE = True
K = 10

results = []

print("Minimal grid search: 4 configurations (~5-15 min)")
print("-" * 60)

for i, config in enumerate(configs, 1):
    rank = config["rank"]
    regParam = config["regParam"]

    print(f"[{i}/{len(configs)}] rank={rank}, regParam={regParam}")
    print("-" * 60)

    try:
        als = ALS(
            userCol="user_index",
            itemCol="item_index",
            ratingCol="rating",
            rank=rank,
            maxIter=ALS_MAX_ITER_TUNE,
            regParam=regParam,
            nonnegative=ALS_NONNEGATIVE,
            implicitPrefs=False,
            coldStartStrategy="drop",
            seed=SEED
        )

        als_model = als.fit(ratings_train_idx)

        pred_scored = als_model.transform(ratings_test_idx)

        df_pred = (
            pred_scored
            .filter(F.col("prediction").isNotNull())
            .select(
                "user_id",
                "item_id",
                "rating",
                F.col("rating").alias("actual_rating"),
                F.when(F.col("prediction") < 1, 1.0)
                 .when(F.col("prediction") > 5, 5.0)
                 .otherwise(F.col("prediction")).alias("prediction")
            )
        )

        rank_window = Window.partitionBy("user_id").orderBy(F.desc("prediction"))

        ranked_pred = (
            df_pred
            .withColumn("rank", F.row_number().over(rank_window))
            .withColumn("hit", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
        )

        topk = ranked_pred.filter(F.col("rank") <= K)

        precision = (
            topk.groupBy("user_id")
            .agg(F.sum("hit").alias("hits"))
            .withColumn("precision_at_k", F.col("hits") / K)
            .agg(F.avg("precision_at_k"))
            .collect()[0][0]
        )

        user_relevant = (
            df_pred
            .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
            .groupBy("user_id")
            .agg(F.sum("relevant").alias("n_relevant"))
        )

        dcg_at_k = (
            topk
            .withColumn(
                "dcg_component",
                F.when(
                    (F.col("hit") == 1) & (F.col("rank") > 0),
                    1 / F.log2(F.col("rank") + 1)
                ).otherwise(F.lit(0.0))
            )
            .groupBy("user_id")
            .agg(F.sum("dcg_component").alias("dcg"))
        )

        ideal_dcg = (
            user_relevant
            .withColumn(
                "ideal_dcg",
                F.expr(f"""
                aggregate(
                    sequence(1, greatest(least(n_relevant,{K}),1)),
                    0D,
                    (acc,x) -> acc + 1/log2(x+1)
                )
                """)
            )
        )

        ndcg = (
            dcg_at_k
            .join(ideal_dcg, on="user_id", how="left")
            .withColumn(
                "ndcg_at_k",
                F.when(
                    (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
                    F.col("dcg") / F.col("ideal_dcg")
                ).otherwise(F.lit(0.0))
            )
            .agg(F.avg("ndcg_at_k"))
            .collect()[0][0]
        )

        results.append({
            "rank": rank,
            "regParam": regParam,
            "ndcg@10": float(ndcg) if ndcg is not None else 0.0,
            "precision@10": float(precision) if precision is not None else 0.0
        })

        print(f"--> NDCG@10: {ndcg:.4f}")
        print(f"--> Precision@10: {precision:.4f}")

    except Exception as e:
        print(f"FAILED: {e}")

print("-" * 60)
print("RESULTS")
print("-" * 60)

results_df = spark.createDataFrame(results).orderBy(F.desc("ndcg@10"))
display(results_df)

best = results_df.first()
print(f"Best config: rank={best['rank']}, regParam={best['regParam']}")
print(f"Best NDCG@10={best['ndcg@10']:.4f}, Precision@10={best['precision@10']:.4f}")

In [0]:
# Retrain with best hyperparameters from grid search
best = results_df.first()

als_final = ALS(
    userCol="user_index",
    itemCol="item_index",
    ratingCol="rating",
    rank=int(best['rank']),
    maxIter=10,
    regParam=float(best['regParam']),
    nonnegative=True,
    implicitPrefs=False,
    coldStartStrategy="drop",
    seed=SEED
)

als_model = als_final.fit(ratings_train_idx)
print(f"Retrained with best config: rank={best['rank']}, regParam={best['regParam']}")

In [0]:
# Inspect learned latent factors
display(als_model.userFactors.limit(10))
display(als_model.itemFactors.limit(10))

In [0]:
# ==========================================
# MF ITEM-ITEM SIMILARITY (dashboard ready)
# ==========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Building MF item-item similarity table...")

# latent vectors from ALS
item_factors = als_model.itemFactors

# attach product names
item_lookup = item_name_lookup

# rename columns for self join
a = (
    item_factors
    .select(
        F.col("id").alias("item_index_a"),
        F.col("features").alias("vec_a")
    )
)

b = (
    item_factors
    .select(
        F.col("id").alias("item_index_b"),
        F.col("features").alias("vec_b")
    )
)

# pair items
pairs = (
    a.crossJoin(b)
    .filter(F.col("item_index_a") < F.col("item_index_b"))
)

# cosine similarity
pairs = (
    pairs
    .withColumn(
        "dot_product",
        F.expr("""
            aggregate(
                zip_with(vec_a, vec_b, (x,y)->x*y),
                0D,
                (acc,x)->acc+x
            )
        """)
    )
    .withColumn(
        "norm_a",
        F.sqrt(F.expr("""
            aggregate(
                transform(vec_a, x->x*x),
                0D,
                (acc,x)->acc+x
            )
        """))
    )
    .withColumn(
        "norm_b",
        F.sqrt(F.expr("""
            aggregate(
                transform(vec_b, x->x*x),
                0D,
                (acc,x)->acc+x
            )
        """))
    )
    .withColumn(
        "similarity",
        F.col("dot_product")/(F.col("norm_a")*F.col("norm_b"))
    )
)

# attach names
mf_item_item = (
    pairs
    .join(
        item_lookup,
        pairs.item_index_a == item_lookup.item_index
    )
    .withColumnRenamed("product_name","item_name")

    .join(
        item_lookup,
        pairs.item_index_b == item_lookup.item_index
    )
    .withColumnRenamed("product_name","similar_item_name")

    .select(
        "item_name",
        "similar_item_name",
        "similarity"
    )
)

# rank top neighbours per item
window_item = Window.partitionBy("item_name").orderBy(F.desc("similarity"))

mf_item_item_topk = (
    mf_item_item
    .withColumn("rank",F.row_number().over(window_item))
    .filter(F.col("rank") <= 20)
)

# save for dashboard
mf_item_item_topk.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.mf_item_item_similarity"
)

print("Saved table: workspace.default.mf_item_item_similarity")

display(mf_item_item_topk.limit(20))

In [0]:
# Holdout scoring for MF on the original 1-5 rating scale
pred_scored = als_model.transform(ratings_test_idx)

df_pred = (
    pred_scored
    .join(
        item_name_lookup.select("item_id", "product_name"),
        on="item_id",
        how="left"
    )
    .select(
        "user_id",
        "item_id",
        "product_name",
        F.col("rating").alias("actual_rating"),
        F.when(F.col("prediction") < 1.0, F.lit(1.0))
         .when(F.col("prediction") > 5.0, F.lit(5.0))
         .otherwise(F.col("prediction"))
         .alias("prediction")
    )
    .withColumn("support", F.lit(0))
)

display(df_pred.limit(20))
print("Scored holdout rows:", df_pred.count())

In [0]:
pred_scored_train = als_model.transform(ratings_train_idx)

df_pred_train = (
    pred_scored_train
    .join(
        item_name_lookup.select("item_id", "product_name"),
        on="item_id",
        how="left"
    )
    .select(
        "user_id",
        "item_id",
        "product_name",
        F.col("rating").alias("actual_rating"),
        F.when(F.col("prediction") < 1.0, F.lit(1.0))
         .when(F.col("prediction") > 5.0, F.lit(5.0))
         .otherwise(F.col("prediction"))
         .alias("prediction")
    )
    .withColumn("support", F.lit(0))
)

display(df_pred_train.limit(20))
print("Scored train rows:", df_pred_train.count())

In [0]:
# Pick a valid user from training data
TARGET_USER_ID = (
    ratings_train_idx
    .select("user_id")
    .distinct()
    .limit(1)
    .collect()[0][0]
)

TOP_N = 10

print("Target user:", TARGET_USER_ID)

target_user_idx = (
    user_index_lookup
    .filter(F.col("user_id") == TARGET_USER_ID)
    .select("user_index")
    .limit(1)
)

seen_items = (
    ratings_train_idx
    .join(target_user_idx, on="user_index", how="inner")
    .select("item_index")
    .distinct()
)

candidate_items = (
    item_index_lookup
    .select("item_index")
    .distinct()
    .join(seen_items, on="item_index", how="left_anti")
)

user_item_candidates = target_user_idx.crossJoin(candidate_items)

target_user_recs = (
    als_model
    .transform(user_item_candidates)
    .filter(F.col("prediction").isNotNull())
    .join(user_index_lookup, on="user_index", how="left")
    .join(item_name_lookup, on="item_index", how="left")
    .select(
        "user_id",
        "item_id",
        "product_name",
        F.col("prediction").alias("predicted_score")
    )
    .orderBy(F.desc("predicted_score"))
    .limit(TOP_N)
)

print("Target user rows:", target_user_idx.count())
print("Seen items:", seen_items.count())
print("Candidate items:", candidate_items.count())
print("Candidate pairs:", user_item_candidates.count())
display(target_user_recs)

In [0]:
# Unity Catalog-safe preview of MF predictions

mf_pred_preview = (
    als_model
    .transform(ratings_test_idx)
    .filter(F.col("prediction").isNotNull())
    .join(
        item_name_lookup.select("item_id", "product_name"),
        on="item_id",
        how="left"
    )
    .select(
        "user_id",
        "item_id",
        "product_name",
        "rating",
        "prediction"
    )
    .orderBy(F.desc("prediction"))
)

display(mf_pred_preview.limit(20))

In [0]:
# Generate Top-K recommendations per user (Unity Catalog safe)

from pyspark.sql.window import Window

K = 10

mf_topk = (
    als_model
    .transform(ratings_test_idx)
    .filter(F.col("prediction").isNotNull())
    .join(
        item_name_lookup.select("item_id", "product_name"),
        on="item_id",
        how="left"
    )
)

# rank predictions per user
window_user = Window.partitionBy("user_id").orderBy(F.desc("prediction"))

mf_topk = (
    mf_topk
    .withColumn("rank", F.row_number().over(window_user))
    .filter(F.col("rank") <= K)
    .select(
        "user_id",
        "item_id",
        "product_name",
        F.col("prediction").alias("predicted_score"),
        "rank"
    )
)

# save to Delta table
mf_topk.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.user_recommendations_mf"
)

print("Saved table: workspace.default.user_recommendations_mf")

display(mf_topk.limit(20))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# RMSE on holdout
rmse = (
    df_pred
    .select(F.sqrt(F.avg(F.pow(F.col("prediction") - F.col("actual_rating"), 2))).alias("rmse"))
    .collect()[0][0]
)

# MAE
mae = (
    df_pred
    .select(F.mean(F.abs(F.col("prediction") - F.col("actual_rating"))).alias("mae"))
    .collect()[0]["mae"]
)

# Ranking metrics from scored holdout pairs
K = 10
rank_window = Window.partitionBy("user_id").orderBy(F.desc("prediction"), F.desc("support"), F.asc("item_id"))

ranked_pred = (
    df_pred
    .withColumn("rank", F.row_number().over(rank_window))
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
)

topk = ranked_pred.filter(F.col("rank") <= K)

user_relevant_counts = (
    df_pred
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
    .groupBy("user_id")
    .agg(F.sum("relevant").alias("n_relevant"))
)

per_user_hits = (
    topk.groupBy("user_id")
    .agg(F.sum("relevant").alias("hits"))
    .join(user_relevant_counts, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn("precision_at_k", F.col("hits") / F.lit(K))
    .withColumn(
        "recall_at_k",
        F.when(F.col("n_relevant") > 0, F.col("hits") / F.col("n_relevant")).otherwise(F.lit(0.0))
    )
)

# MAP@K
ap_base = (
    topk
    .withColumn("cum_relevant", F.sum("relevant").over(rank_window.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn(
        "precision_at_rank",
        F.when(F.col("relevant") == 1, F.col("cum_relevant") / F.col("rank")).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("precision_at_rank").alias("sum_precisions"))
    .join(user_relevant_counts, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn(
        "ap_at_k",
        F.when(F.col("n_relevant") > 0, F.col("sum_precisions") / F.least(F.col("n_relevant"), F.lit(K)))
         .otherwise(F.lit(0.0))
    )
)

precision_at_k = per_user_hits.agg(F.avg("precision_at_k")).collect()[0][0]
recall_at_k = per_user_hits.agg(F.avg("recall_at_k")).collect()[0][0]
map_at_k = ap_base.agg(F.avg("ap_at_k")).collect()[0][0]

# Catalogue coverage against the full raw catalogue, not the filtered train/test subset
total_catalog_items = ratings_clean.select("item_id").distinct().count()

catalogue_coverage = (
    topk.select("item_id").distinct().count() / total_catalog_items
    if total_catalog_items > 0 else 0.0
)

# NDCG@K (safe version)
dcg_base = (
    topk
    .withColumn(
        "dcg_component",
        F.when(
            (F.col("relevant") == 1) & (F.col("rank") > 0),
            1 / F.log2(F.col("rank") + 1)
        ).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("dcg_component").alias("dcg"))
)

ideal_dcg = (
    user_relevant_counts
    .withColumn(
        "ideal_dcg",
        F.expr(f'''
        aggregate(
            sequence(1, greatest(least(n_relevant,{K}),1)),
            0D,
            (acc,x) -> acc + 1/log2(x+1)
        )
        ''')
    )
)

ndcg_df = (
    dcg_base
    .join(ideal_dcg, on="user_id", how="left")
    .withColumn(
        "ndcg_at_k",
        F.when(
            (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
            F.col("dcg") / F.col("ideal_dcg")
        ).otherwise(F.lit(0.0))
    )
)

ndcg_at_k = ndcg_df.select(F.avg("ndcg_at_k")).collect()[0][0]

random_hit_rate = df_pred.select(F.avg(F.when(F.col("actual_rating") >= 4.0, 1.0).otherwise(0.0))).collect()[0][0]
lift_over_random = precision_at_k / random_hit_rate if random_hit_rate and random_hit_rate > 0 else None

# Inter-user diversity: 1 - average Jaccard overlap across top-K item lists
user_lists = topk.groupBy("user_id").agg(F.collect_set("item_id").alias("rec_list"))
user_pairs = (
    user_lists.alias("a")
    .crossJoin(user_lists.alias("b"))
    .filter(F.col("a.user_id") < F.col("b.user_id"))
    .select(
        F.size(F.array_intersect(F.col("a.rec_list"), F.col("b.rec_list"))).alias("intersect_size"),
        F.size(F.array_union(F.col("a.rec_list"), F.col("b.rec_list"))).alias("union_size")
    )
    .withColumn(
        "inter_user_diversity",
        F.when(F.col("union_size") > 0, 1 - (F.col("intersect_size") / F.col("union_size"))).otherwise(F.lit(0.0))
    )
)
inter_user_diversity = user_pairs.agg(F.avg("inter_user_diversity")).collect()[0][0]

# Serendipity proxy: relevant recommendations that are not among the most popular training items
item_popularity = ratings.groupBy("item_id").agg(F.count("*").alias("popularity"))
pop_threshold = item_popularity.approxQuantile("popularity", [0.8], 0.01)[0] if item_popularity.count() > 0 else 0

serendipity = (
    topk.join(item_popularity, on="item_id", how="left")
    .withColumn("unexpected", F.when(F.col("popularity") <= pop_threshold, 1).otherwise(0))
    .withColumn("serendipitous", F.when((F.col("relevant") == 1) & (F.col("unexpected") == 1), 1).otherwise(0))
    .groupBy("user_id")
    .agg((F.sum("serendipitous") / F.lit(K)).alias("serendipity"))
)
serendipity_score = serendipity.agg(F.avg("serendipity")).collect()[0][0]

metrics_df = spark.createDataFrame([
    ("RMSE", float(rmse) if rmse is not None else None),
    ("MAE", float(mae) if mae is not None else None),
    (f"Precision@{K}", float(precision_at_k) if precision_at_k is not None else None),
    (f"Recall@{K}", float(recall_at_k) if recall_at_k is not None else None),
    (f"MAP@{K}", float(map_at_k) if map_at_k is not None else None),
    (f"NDCG@{K}", float(ndcg_at_k) if ndcg_at_k is not None else None),
    ("Catalogue coverage", float(catalogue_coverage) if catalogue_coverage is not None else None),
    ("Lift over random", float(lift_over_random) if lift_over_random is not None else None),
    ("Serendipity", float(serendipity_score) if serendipity_score is not None else None),
    ("Inter-user diversity", float(inter_user_diversity) if inter_user_diversity is not None else None),
], ["metric", "value"])

display(metrics_df)

In [0]:
# RMSE on train
rmse_train = (
    df_pred_train
    .select(F.sqrt(F.avg(F.pow(F.col("prediction") - F.col("actual_rating"), 2))).alias("rmse"))
    .collect()[0][0]
)

# MAE
mae_train = (
    df_pred_train
    .select(F.mean(F.abs(F.col("prediction") - F.col("actual_rating"))).alias("mae"))
    .collect()[0]["mae"]
)

ranked_pred_train = (
    df_pred_train
    .withColumn("rank", F.row_number().over(rank_window))
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
)

topk_train = ranked_pred_train.filter(F.col("rank") <= K)

user_relevant_counts_train = (
    df_pred_train
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
    .groupBy("user_id")
    .agg(F.sum("relevant").alias("n_relevant"))
)

per_user_hits_train = (
    topk_train.groupBy("user_id")
    .agg(F.sum("relevant").alias("hits"))
    .join(user_relevant_counts_train, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn("precision_at_k", F.col("hits") / F.lit(K))
    .withColumn(
        "recall_at_k",
        F.when(F.col("n_relevant") > 0, F.col("hits") / F.col("n_relevant")).otherwise(F.lit(0.0))
    )
)

# MAP@K
ap_base_train = (
    topk_train
    .withColumn("cum_relevant", F.sum("relevant").over(rank_window.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn(
        "precision_at_rank",
        F.when(F.col("relevant") == 1, F.col("cum_relevant") / F.col("rank")).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("precision_at_rank").alias("sum_precisions"))
    .join(user_relevant_counts_train, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn(
        "ap_at_k",
        F.when(F.col("n_relevant") > 0, F.col("sum_precisions") / F.least(F.col("n_relevant"), F.lit(K)))
         .otherwise(F.lit(0.0))
    )
)

precision_at_k_train = per_user_hits_train.agg(F.avg("precision_at_k")).collect()[0][0]
recall_at_k_train = per_user_hits_train.agg(F.avg("recall_at_k")).collect()[0][0]
map_at_k_train = ap_base_train.agg(F.avg("ap_at_k")).collect()[0][0]

catalogue_coverage_train = (
    topk_train.select("item_id").distinct().count() / total_catalog_items
    if total_catalog_items > 0 else 0.0
)

# NDCG@K (safe version)
dcg_base_train = (
    topk_train
    .withColumn(
        "dcg_component",
        F.when(
            (F.col("relevant") == 1) & (F.col("rank") > 0),
            1 / F.log2(F.col("rank") + 1)
        ).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("dcg_component").alias("dcg"))
)

ideal_dcg_train = (
    user_relevant_counts_train
    .withColumn(
        "ideal_dcg",
        F.expr(f'''
        aggregate(
            sequence(1, greatest(least(n_relevant,{K}),1)),
            0D,
            (acc,x) -> acc + 1/log2(x+1)
        )
        ''')
    )
)

ndcg_df_train = (
    dcg_base_train
    .join(ideal_dcg_train, on="user_id", how="left")
    .withColumn(
        "ndcg_at_k",
        F.when(
            (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
            F.col("dcg") / F.col("ideal_dcg")
        ).otherwise(F.lit(0.0))
    )
)

ndcg_at_k_train = ndcg_df_train.select(F.avg("ndcg_at_k")).collect()[0][0]

random_hit_rate_train = df_pred_train.select(F.avg(F.when(F.col("actual_rating") >= 4.0, 1.0).otherwise(0.0))).collect()[0][0]
lift_over_random_train = precision_at_k_train / random_hit_rate_train if random_hit_rate_train and random_hit_rate_train > 0 else None

# Inter-user diversity: 1 - average Jaccard overlap across top-K item lists
user_lists_train = topk_train.groupBy("user_id").agg(F.collect_set("item_id").alias("rec_list"))
user_pairs_train = (
    user_lists_train.alias("a")
    .crossJoin(user_lists_train.alias("b"))
    .filter(F.col("a.user_id") < F.col("b.user_id"))
    .select(
        F.size(F.array_intersect(F.col("a.rec_list"), F.col("b.rec_list"))).alias("intersect_size"),
        F.size(F.array_union(F.col("a.rec_list"), F.col("b.rec_list"))).alias("union_size")
    )
    .withColumn(
        "inter_user_diversity",
        F.when(F.col("union_size") > 0, 1 - (F.col("intersect_size") / F.col("union_size"))).otherwise(F.lit(0.0))
    )
)
inter_user_diversity_train = user_pairs_train.agg(F.avg("inter_user_diversity")).collect()[0][0]

serendipity_train = (
    topk_train.join(item_popularity, on="item_id", how="left")
    .withColumn("unexpected", F.when(F.col("popularity") <= pop_threshold, 1).otherwise(0))
    .withColumn("serendipitous", F.when((F.col("relevant") == 1) & (F.col("unexpected") == 1), 1).otherwise(0))
    .groupBy("user_id")
    .agg((F.sum("serendipitous") / F.lit(K)).alias("serendipity"))
)
serendipity_score_train = serendipity_train.agg(F.avg("serendipity")).collect()[0][0]

metrics_df_train = spark.createDataFrame([
    ("RMSE", float(rmse_train) if rmse_train is not None else None),
    ("MAE", float(mae_train) if mae_train is not None else None),
    (f"Precision@{K}", float(precision_at_k_train) if precision_at_k_train is not None else None),
    (f"Recall@{K}", float(recall_at_k_train) if recall_at_k_train is not None else None),
    (f"MAP@{K}", float(map_at_k_train) if map_at_k_train is not None else None),
    (f"NDCG@{K}", float(ndcg_at_k_train) if ndcg_at_k_train is not None else None),
    ("Catalogue coverage", float(catalogue_coverage_train) if catalogue_coverage_train is not None else None),
    ("Lift over random", float(lift_over_random_train) if lift_over_random_train is not None else None),
    ("Serendipity", float(serendipity_score_train) if serendipity_score_train is not None else None),
    ("Inter-user diversity", float(inter_user_diversity_train) if inter_user_diversity_train is not None else None),
], ["metric", "value"])

display(metrics_df_train)

In [0]:
metric_data = [
{"model_name": "Matrix Factorization",
 "dataset": "Test",
 "rmse": (float(rmse) if rmse is not None else None),
 "mae": (float(mae) if mae is not None else None),
 "precision": (float(precision_at_k) if precision_at_k is not None else None),
 "recall": (float(recall_at_k) if recall_at_k is not None else None),
 "map": (float(map_at_k) if map_at_k is not None else None),
 "ndcg": (float(ndcg_at_k) if ndcg_at_k is not None else None),
 "coverage": (float(catalogue_coverage) if catalogue_coverage is not None else None),
 "lift": (float(lift_over_random) if lift_over_random is not None else None),
 "serendipity": (float(serendipity_score) if serendipity_score is not None else None),
 "diversity": (float(inter_user_diversity) if inter_user_diversity is not None else None),
 "k_value": K
}
]

metric_data_train = [
{"model_name": "Matrix Factorization",
 "dataset": "Train",
 "rmse": (float(rmse_train) if rmse_train is not None else None),
 "mae": (float(mae_train) if mae_train is not None else None),
 "precision": (float(precision_at_k_train) if precision_at_k_train is not None else None),
 "recall": (float(recall_at_k_train) if recall_at_k_train is not None else None),
 "map": (float(map_at_k_train) if map_at_k_train is not None else None),
 "ndcg": (float(ndcg_at_k_train) if ndcg_at_k_train is not None else None),
 "coverage": (float(catalogue_coverage_train) if catalogue_coverage_train is not None else None),
 "lift": (float(lift_over_random_train) if lift_over_random_train is not None else None),
 "serendipity": (float(serendipity_score_train) if serendipity_score_train is not None else None),
 "diversity": (float(inter_user_diversity_train) if inter_user_diversity_train is not None else None),
 "k_value": K
}
]

model_metrics = spark.createDataFrame(metric_data).withColumn("timestamp", F.current_timestamp())
model_metrics_train = spark.createDataFrame(metric_data_train).withColumn("timestamp", F.current_timestamp())
model_metrics.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.model_metrics")

model_metrics_train.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.model_metrics")

model_eval = spark.table("workspace.default.model_metrics")
display(model_eval)


Save Model to Unity Catalog

In [0]:
import mlflow
import mlflow.spark

# Explicitly set experiment so you know exactly where to look
mlflow.set_experiment("/Shared/mf_experiment")

# Start MLflow run and log ALS modelnm
input_example = ratings_train_idx.limit(1).toPandas()
with mlflow.start_run(run_name="ALS Matrix Factorization"):
    mlflow.spark.log_model(
        als_model, 
        "als_model", 
        registered_model_name="workspace.default.als_model_uc",
        dfs_tmpdir="/Volumes/workspace/default/sephora_surfers",
        input_example = input_example
    )
    
    # Log metrics for test set
    for row in model_metrics.collect():
        mlflow.log_metric("rmse_test", row.rmse)
        mlflow.log_metric("mae_test", row.mae)
        mlflow.log_metric("precision_test", row.precision)
        mlflow.log_metric("recall_test", row.recall)
        mlflow.log_metric("map_test", row.map)
        mlflow.log_metric("ndcg_test", row.ndcg)
        mlflow.log_metric("coverage_test", row.coverage)
        mlflow.log_metric("lift_test", row.lift)
        mlflow.log_metric("serendipity_test", row.serendipity)
        mlflow.log_metric("diversity_test", row.diversity)
        mlflow.log_param("k_value_test", row.k_value)
    
    # Log metrics for train set
    for row in model_metrics_train.collect():
        mlflow.log_metric("rmse_train", row.rmse)
        mlflow.log_metric("mae_train", row.mae)
        mlflow.log_metric("precision_train", row.precision)
        mlflow.log_metric("recall_train", row.recall)
        mlflow.log_metric("map_train", row.map)
        mlflow.log_metric("ndcg_train", row.ndcg)
        mlflow.log_metric("coverage_train", row.coverage)
        mlflow.log_metric("lift_train", row.lift)
        mlflow.log_metric("serendipity_train", row.serendipity)
        mlflow.log_metric("diversity_train", row.diversity)
        mlflow.log_param("k_value_train", row.k_value)

In [0]:
from mlflow.tracking import MlflowClient
import mlflow.spark

MODEL_NAME = "workspace.default.als_model_uc"
TMP_DIR = "/Volumes/workspace/default/sephora_surfers"

client = MlflowClient()

# get latest version number manually
versions = client.search_model_versions(f"name='{MODEL_NAME}'")

latest_version = max([int(v.version) for v in versions])

print("Latest version:", latest_version)

# load latest model
model_version_uri = f"models:/{MODEL_NAME}/{latest_version}"

loaded_model = mlflow.spark.load_model(
    model_version_uri,
    dfs_tmpdir=TMP_DIR
)

predictions = loaded_model.transform(ratings_test_idx)

display(predictions)